In [0]:
# Import SparkSession - entry point for PySpark
from pyspark.sql import SparkSession

# Import functions for DataFrame operations and SQL queries
from pyspark.sql.functions import (
    col, lit, when, expr, sum, avg, count, desc,
    to_date, year, month, dayofmonth, datediff,
    to_timestamp, lag, lead, unix_timestamp, date_format,
    window, broadcast
)

# Import Window for window functions used in time series and advanced analytics
from pyspark.sql.window import Window

# Import data types to define schema explicitly if needed
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

# MLlib imports for feature engineering, regression, classification, clustering, evaluation, and hyperparameter tuning
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Import storage level to control caching/persisting
from pyspark.storagelevel import StorageLevel

# For data visualization outside Spark (convert small samples to pandas)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# For testing PySpark code with unittest framework
import unittest

# Initialize SparkSession (example)
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()


In [0]:
print("welcome")

In [0]:
# Load the data from the saved table
df = spark.table("superstore_sales_dataset")

# Show the first 10 rows of the DataFrame
df.show(2)


In [0]:
# Show the first 2 rows of the DataFrame
df.show(2)

In [0]:
# Print the schema (column names and data types)
df.printSchema()

# Show summary statistics for numeric columns
df.describe().show()

In [0]:
#Data Cleaning (drop duplicates, drop nulls, modify schema)
from pyspark.sql.functions import col

# Drop duplicate rows
df_cleaned = df.dropDuplicates()

# Drop rows with any null values
df_cleaned = df_cleaned.dropna()

# Cast specific columns to appropriate data types
df_cleaned = df_cleaned.withColumn("Sales", col("Sales").cast("double")) \
                       .withColumn("Postal Code", col("Postal Code").cast("string")) \
                       .withColumn("Order Date", col("Order Date").cast("date")) \
                       .withColumn("Ship Date", col("Ship Date").cast("date"))

# Verify schema after type casting
df_cleaned.printSchema()

# Show a few cleaned rows
df_cleaned.show(5)


In [0]:
from pyspark.sql.functions import sum, desc
# Top 10 customers by total sales
top_customers = df_cleaned.groupBy("Customer Name") \
                          .agg(sum("Sales").alias("Total_Sales")) \
                          .orderBy(desc("Total_Sales")) \
                          .limit(10)

top_customers.show()

In [0]:
# Total sales by Region
df_cleaned.groupBy("Region") \
          .agg(sum("Sales").alias("Total_Sales")) \
          .orderBy(desc("Total_Sales")) \
          .show()

# Total sales by State
df_cleaned.groupBy("State") \
          .agg(sum("Sales").alias("Total_Sales")) \
          .orderBy(desc("Total_Sales")) \
          .show()

# Total sales by City
df_cleaned.groupBy("City") \
          .agg(sum("Sales").alias("Total_Sales")) \
          .orderBy(desc("Total_Sales")) \
          .show()

# Total sales by Country (in case there's more than one)
df_cleaned.groupBy("Country") \
          .agg(sum("Sales").alias("Total_Sales")) \
          .orderBy(desc("Total_Sales")) \
          .show()


In [0]:
from pyspark.sql.functions import year, month, quarter

# Add time-based columns
df_time = df_cleaned.withColumn("Year", year("Order Date")) \
                    .withColumn("Month", month("Order Date")) \
                    .withColumn("Quarter", quarter("Order Date"))

# Yearly sales
df_time.groupBy("Year") \
       .agg(sum("Sales").alias("Yearly_Sales")) \
       .orderBy("Year") \
       .show()

# Monthly sales
df_time.groupBy("Year", "Month") \
       .agg(sum("Sales").alias("Monthly_Sales")) \
       .orderBy("Year", "Month") \
       .show()

# Quarterly sales
df_time.groupBy("Year", "Quarter") \
       .agg(sum("Sales").alias("Quarterly_Sales")) \
       .orderBy("Year", "Quarter") \
       .show()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Define window by Region, ordered by sales descending
window_spec = Window.partitionBy("Region").orderBy(desc("Total_Sales"))

# Calculate total sales per customer per region
customer_sales_region = df_cleaned.groupBy("Region", "Customer Name") \
                                  .agg(sum("Sales").alias("Total_Sales"))

# Apply ranking
ranked_customers = customer_sales_region.withColumn("Rank", rank().over(window_spec))

# Filter to get top 3 customers per region
top_customers_per_region = ranked_customers.filter("Rank <= 3")

top_customers_per_region.show()


In [0]:
from pyspark.sql.functions import min

# First order date per customer
first_order = df_cleaned.groupBy("Customer Name") \
                        .agg(min("Order Date").alias("First_Order_Date")) \
                        .orderBy("First_Order_Date")

first_order.show()


In [0]:
from pyspark.sql.functions import date_format
import pandas as pd

# Group by Year-Month and aggregate sales
monthly_sales = df_cleaned.withColumn("YearMonth", date_format("Order Date", "yyyy-MM")) \
                          .groupBy("YearMonth") \
                          .agg(sum("Sales").alias("Total_Sales")) \
                          .orderBy("YearMonth")

# Convert to Pandas for time series modeling
monthly_sales_pd = monthly_sales.toPandas()

# Convert YearMonth to datetime type and set as index
monthly_sales_pd['YearMonth'] = pd.to_datetime(monthly_sales_pd['YearMonth'])
monthly_sales_pd.set_index('YearMonth', inplace=True)

monthly_sales_pd.head()


In [0]:
print("hello")